In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd


/opt/anaconda3/envs/springIWenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import string

In [5]:
trans_table = str.maketrans('', '', string.punctuation)

In [3]:
multi_lingual_model = SentenceTransformer('sentence-transformers/LaBSE')
raw_verses = pd.read_csv('all_data_hebrew_numbering.csv')

In [9]:
def clean_score(score):
    return round(float(score), 5)

def compute_similarity(model, list1, list2, list3, list4):
    scores_1_2 = []
    scores_1_3 = []
    scores_1_4 = []
    scores_2_3 = []
    scores_2_4 = []
    scores_3_4 = []
    count = 0
    for raw_verse1, raw_verse2, raw_verse3, raw_verse4 in zip(list1, list2, list3, list4):

        verse1 = raw_verse1.translate(trans_table)
        verse2 = raw_verse2.translate(trans_table)
        verse3 = raw_verse3.translate(trans_table)
        verse4 = raw_verse4.translate(trans_table)


        embeddings = model.encode([verse1, verse2, verse3, verse4])
        similarities = model.similarity(embeddings, embeddings)
        scores_1_2.append(clean_score(similarities[1][0]))
        scores_1_3.append(clean_score(similarities[2][0]))
        scores_1_4.append(clean_score(similarities[3][0]))
        scores_2_3.append(clean_score(similarities[2][1]))
        scores_2_4.append(clean_score(similarities[3][1]))
        scores_3_4.append(clean_score(similarities[3][2]))
        count+=1
        if (count % 50 == 0):
            print(count)

    return scores_1_2, scores_1_3, scores_1_4, scores_2_3, scores_2_4, scores_3_4

In [7]:
raw_hebrew = raw_verses['Hebrew'][:500]
raw_septuagint = raw_verses['Septuagint'][:500]
raw_gallican_psalter = raw_verses['Gallican Psalter'][:500]
raw_hebrew_psalter = raw_verses['Hebrew Psalter'][:500]

In [10]:
m_heb_sept,m_heb_pgal,m_heb_pheb, m_sept_pgal, m_sept_pheb, m_pgal_pheb = compute_similarity(multi_lingual_model,
    raw_hebrew, raw_septuagint, raw_gallican_psalter, raw_hebrew_psalter)

50
100
150
200
250
300
350
400
450
500


In [11]:
raw_verses_curr = raw_verses[:500].copy()

In [13]:
raw_verses_curr['heb_sept'] = m_heb_sept
raw_verses_curr['heb_pgal'] = m_heb_pgal
raw_verses_curr['heb_pheb'] = m_heb_pheb
raw_verses_curr['sept_pgal'] = m_sept_pgal
raw_verses_curr['sept_pheb'] = m_sept_pheb
raw_verses_curr['pgal_pheb'] = m_pgal_pheb

In [15]:
raw_verses_curr.to_csv('labse_result1.csv')